# Ajuste fino con LoRA sobre modelo NLI multilingue, cinco clases

Proyecto 2, curso CC0C2. Corpus: Codigo Civil Peruano. Clases: contradiccion, compatibilidad, especificacion, excepcion, ausencia de relacion.

Este cuaderno reemplaza al intento anterior, que aplico LoRA sobre un modelo NLI entrenado exclusivamente en ingles (cross-encoder/nli-deberta-v3-large) con un dataset sintetico de oraciones cortas, y obtuvo F1 de 0 en el gold set real. El cambio respecto a esa version es doble: se usa un modelo NLI multilingue con cobertura de espanol, y un conjunto de entrenamiento construido a partir de articulos reales del Codigo Civil, en lugar de oraciones inventadas de longitud y estilo distintos al dominio.

Requiere entorno de ejecucion con GPU. En Colab: Entorno de ejecucion, Cambiar tipo de entorno de ejecucion, GPU (T4).


In [1]:
!pip install -q transformers peft accelerate datasets scikit-learn "pandas==2.2.2"
!pip install -q -U torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.0 MB/s eta 0:00:00


In [2]:
import json
import os
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", DEVICE)
if DEVICE == "cpu":
    print("no se detecto GPU, revisar configuracion del entorno de ejecucion")


dispositivo: cuda


## Datos

Se necesitan dos archivos:

- dataset_5clases_entrenamiento.json, con 211 pares de entrenamiento construidos sobre articulos reales del corpus.
- gold_dataset_5clases_evaluacion.json, que contiene el campo etiqueta_final con las 5 clases originales para 142 pares de evaluacion, sin superposicion con el conjunto de entrenamiento.


In [3]:
ETIQUETAS = ["CONTRADICCION", "COMPATIBILIDAD", "ESPECIFICACION", "EXCEPCION", "NO_RELACION"]
etiqueta_a_id = {e: i for i, e in enumerate(ETIQUETAS)}
id_a_etiqueta = {i: e for i, e in enumerate(ETIQUETAS)}

with open("dataset_5clases_entrenamiento.json", "r", encoding="utf-8") as f:
    _raw = json.load(f)
train_data = _raw["dataset"]

for item in train_data:
    item["label_id"] = etiqueta_a_id[item["etiqueta"]]

print("pares de entrenamiento:", len(train_data))
print(Counter(item["etiqueta"] for item in train_data))


pares de entrenamiento: 211
Counter({'COMPATIBILIDAD': 63, 'NO_RELACION': 45, 'EXCEPCION': 38, 'ESPECIFICACION': 35, 'CONTRADICCION': 30})


In [6]:
with open("gold_dataset_5clases_evaluacion.json", "r", encoding="utf-8") as f:
    _raw_gold = json.load(f)
gold_data = _raw_gold["gold"]

for item in gold_data:
    item["label_id"] = etiqueta_a_id[item["etiqueta_final"]]

print("pares de evaluacion:", len(gold_data))
print(Counter(item["etiqueta_final"] for item in gold_data))


pares de evaluacion: 142
Counter({'NO_RELACION': 45, 'ESPECIFICACION': 35, 'COMPATIBILIDAD': 32, 'EXCEPCION': 20, 'CONTRADICCION': 10})


In [7]:
ids_entrenamiento = set(item.get("articulo_a") for item in train_data)
ids_gold = set(str(item.get("articulo_a")) for item in gold_data) | set(str(item.get("articulo_b")) for item in gold_data)
interseccion = set(str(x) for x in ids_entrenamiento) & ids_gold
print("articulos del conjunto de entrenamiento que tambien aparecen en el gold set:", len(interseccion))


articulos del conjunto de entrenamiento que tambien aparecen en el gold set: 16


## Modelo base y configuracion de LoRA

Se parte de un modelo NLI multilingue preentrenado (MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7), y se reemplaza su cabeza de clasificacion original de tres salidas por una nueva de cinco salidas, ya que las clases de este proyecto no coinciden con las clases nativas de NLI. Los pesos del encoder se mantienen congelados salvo por los adaptadores LoRA, que se entrenan junto con la nueva cabeza.


In [8]:
MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

modelo_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    ignore_mismatched_sizes=True,
)

nombres_modulos = set()
for nombre, modulo in modelo_base.named_modules():
    if "Linear" in modulo.__class__.__name__:
        nombres_modulos.add(nombre.split(".")[-1])
print("nombres de capas lineales encontradas:", sorted(nombres_modulos))


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

...

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `3`.


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([5, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


nombres de capas lineales encontradas: ['classifier', 'dense', 'key_proj', 'query_proj', 'value_proj']


In [9]:
LORA_CONFIG = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"],
)

def construir_modelo():
    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=5,
        ignore_mismatched_sizes=True,
    )
    modelo = get_peft_model(modelo, LORA_CONFIG)
    return modelo.to(DEVICE)

modelo_prueba = construir_modelo()
modelo_prueba.print_trainable_parameters()
del modelo_prueba


[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([5, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


trainable params: 298,757 || all params: 279,111,946 || trainable%: 0.1070


## Preparacion de los datos para entrenamiento

Se construye una clase de dataset de PyTorch que tokeniza cada par (texto_a, texto_b) como una entrada de dos segmentos, siguiendo el formato estandar de un modelo NLI.


In [10]:
class ParesDataset(Dataset):
    def __init__(self, items, tokenizer, max_length=384):
        self.items = items
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        texto_a = item.get("text_a") or item.get("texto_a")
        texto_b = item.get("text_b") or item.get("texto_b")
        enc = self.tokenizer(
            texto_a,
            texto_b,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
        )
        enc = {k: torch.tensor(v) for k, v in enc.items()}
        enc["labels"] = torch.tensor(item["label_id"])
        return enc


In [11]:
def calcular_metricas(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }


In [12]:
from transformers import Trainer
import torch.nn as nn

conteo_clases = Counter(item["label_id"] for item in train_data)
total = len(train_data)
num_clases = 5
pesos_clase = torch.tensor(
    [total / (num_clases * conteo_clases[i]) for i in range(num_clases)],
    dtype=torch.float,
).to(DEVICE)

print("pesos por clase:", {ETIQUETAS[i]: round(float(pesos_clase[i]), 3) for i in range(num_clases)})


class TrainerConPesos(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        funcion_perdida = nn.CrossEntropyLoss(weight=pesos_clase)
        loss = funcion_perdida(logits, labels)
        return (loss, outputs) if return_outputs else loss

pesos por clase: {'CONTRADICCION': 1.407, 'COMPATIBILIDAD': 0.67, 'ESPECIFICACION': 1.206, 'EXCEPCION': 1.111, 'NO_RELACION': 0.938}


## Validacion cruzada estratificada

Dado el tamano reducido del conjunto de entrenamiento, se utiliza validacion cruzada estratificada de 5 particiones en lugar de una unica division entrenamiento y validacion. En cada particion se construye un modelo nuevo con adaptadores LoRA inicializados de cero, para evitar que el resultado de una particion dependa del entrenamiento de otra.


In [13]:
EPOCAS = 6
LOTE = 8
TASA_APRENDIZAJE = 2e-4

labels_para_split = [item["label_id"] for item in train_data]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados_folds = []

for fold, (idx_train, idx_val) in enumerate(skf.split(train_data, labels_para_split)):
    print(f"particion {fold + 1} de 5")

    items_train = [train_data[i] for i in idx_train]
    items_val = [train_data[i] for i in idx_val]

    ds_train = ParesDataset(items_train, tokenizer)
    ds_val = ParesDataset(items_val, tokenizer)

    modelo = construir_modelo()

    args = TrainingArguments(
        output_dir=f"./salida_fold_{fold}",
        num_train_epochs=EPOCAS,
        per_device_train_batch_size=LOTE,
        per_device_eval_batch_size=LOTE,
        learning_rate=TASA_APRENDIZAJE,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=20,
        report_to="none",
        fp16=False,
    )

    trainer = TrainerConPesos(
      model=modelo,
      args=args,
      train_dataset=ds_train,
      eval_dataset=ds_val,
      compute_metrics=calcular_metricas,
  )

    trainer.train()
    metricas = trainer.evaluate()
    resultados_folds.append(metricas)
    print(metricas)

    del modelo, trainer
    torch.cuda.empty_cache()


particion 1 de 5


[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([5, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.554080,1.460616,0.348837,0.336276
2,1.383272,1.389980,0.441860,0.418624
3,1.344535,1.349324,0.441860,0.392545
4,1.249781,1.355067,0.418605,0.389768
5,1.179327,1.341327,0.441860,0.426909
6,1.203547,1.342616,0.418605,0.394292


...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([5, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.534555,1.351895,0.333333,0.284334
2,1.438059,1.288800,0.380952,0.319042
3,1.333411,1.253611,0.333333,0.324803
4,1.260830,1.240149,0.380952,0.352208
5,1.176205,1.224877,0.333333,0.326061
6,1.287396,1.224618,0.333333,0.326061


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
1.287396,1.224618,6,0.333333,0.326061


{'eval_loss': 1.2246179580688477, 'eval_accuracy': 0.3333333333333333, 'eval_f1_macro': 0.32606060606060605}


In [14]:
acc_folds = [r["eval_accuracy"] for r in resultados_folds]
f1_folds = [r["eval_f1_macro"] for r in resultados_folds]

print("accuracy promedio:", np.mean(acc_folds), "desviacion:", np.std(acc_folds))
print("f1 macro promedio:", np.mean(f1_folds), "desviacion:", np.std(f1_folds))


accuracy promedio: 0.3694352159468438 desviacion: 0.03584243921257783
f1 macro promedio: 0.34320815142042765 desviacion: 0.052683967310061794


## Entrenamiento final

Con el numero de epocas usado en la validacion cruzada, se entrena un modelo final sobre la totalidad del conjunto de 211 pares, que es el que se evalua contra el gold set y el que se guarda para su uso posterior.


In [15]:
ds_train_completo = ParesDataset(train_data, tokenizer)

modelo_final = construir_modelo()

args_final = TrainingArguments(
    output_dir="./modelo_final",
    num_train_epochs=EPOCAS,
    per_device_train_batch_size=LOTE,
    learning_rate=TASA_APRENDIZAJE,
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    fp16=False,
)

trainer_final = TrainerConPesos(
    model=modelo_final,
    args=args_final,
    train_dataset=ds_train_completo,
    compute_metrics=calcular_metricas,
)

trainer_final.train()


[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([5, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Step,Training Loss
20,1.505194
40,1.393323
60,1.398702
80,1.255753
100,1.237934
120,1.221775
140,1.244140
160,1.197166


TrainOutput(global_step=162, training_loss=1.308270784071934, metrics={'train_runtime': 39.6263, 'train_samples_per_second': 31.948, 'train_steps_per_second': 4.088, 'total_flos': 250706590110720.0, 'train_loss': 1.308270784071934, 'epoch': 6.0})

## Evaluacion sobre el gold set

Se evalua el modelo final sobre los 142 pares del gold set, que no se usaron en ningun momento del entrenamiento ni de la validacion cruzada.


In [16]:
ds_gold = ParesDataset(gold_data, tokenizer)
prediccion_gold = trainer_final.predict(ds_gold)

y_true = prediccion_gold.label_ids
y_pred = np.argmax(prediccion_gold.predictions, axis=1)

print("accuracy en gold set:", accuracy_score(y_true, y_pred))
print("f1 macro en gold set:", f1_score(y_true, y_pred, average="macro", zero_division=0))
print()
print(classification_report(y_true, y_pred, target_names=ETIQUETAS, zero_division=0))


accuracy en gold set: 0.2605633802816901
f1 macro en gold set: 0.21245458960660515

                precision    recall  f1-score   support

 CONTRADICCION       0.20      0.20      0.20        10
COMPATIBILIDAD       0.28      0.34      0.31        32
ESPECIFICACION       0.00      0.00      0.00        35
     EXCEPCION       0.13      0.25      0.17        20
   NO_RELACION       0.35      0.42      0.38        45

      accuracy                           0.26       142
     macro avg       0.19      0.24      0.21       142
  weighted avg       0.21      0.26      0.23       142



In [17]:
matriz = confusion_matrix(y_true, y_pred, labels=list(range(5)))
df_matriz = pd.DataFrame(matriz, index=ETIQUETAS, columns=ETIQUETAS)
print(df_matriz)


                CONTRADICCION  COMPATIBILIDAD  ESPECIFICACION  EXCEPCION  \
CONTRADICCION               2               1               0          6   
COMPATIBILIDAD              2              11               0          4   
ESPECIFICACION              2              10               0          9   
EXCEPCION                   3               6               0          5   
NO_RELACION                 1              11               0         14   

                NO_RELACION  
CONTRADICCION             1  
COMPATIBILIDAD           15  
ESPECIFICACION           14  
EXCEPCION                 6  
NO_RELACION              19  


## Comparacion contra el modelo sin ajuste fino

Para saber si el ajuste fino aporto algo, se compara contra el mismo modelo evaluado en modo zero shot, es decir, sin adaptadores LoRA, sobre el mismo gold set. El modelo base tiene tres salidas nativas, asi que para esta comparacion se agrupan las cinco clases del gold set en tres, siguiendo el mismo criterio usado en el resto del proyecto: contradiccion y excepcion se agrupan como contradiccion, especificacion como implicacion, compatibilidad y ausencia de relacion como neutral.


In [18]:
from transformers import pipeline

MAPEO_3 = {
    "CONTRADICCION": "contradiction",
    "EXCEPCION": "contradiction",
    "ESPECIFICACION": "entailment",
    "COMPATIBILIDAD": "neutral",
    "NO_RELACION": "neutral",
}

modelo_zero_shot = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
id2label_base = {i: l.lower() for i, l in modelo_zero_shot.config.id2label.items()}

y_true_3 = []
y_pred_3 = []

modelo_zero_shot.eval()
with torch.no_grad():
    for item in gold_data:
        texto_a = item["texto_a"]
        texto_b = item["texto_b"]
        enc = tokenizer(texto_a, texto_b, truncation=True, max_length=384, return_tensors="pt").to(DEVICE)
        salida = modelo_zero_shot(**enc).logits.softmax(dim=-1)[0].cpu().numpy()
        etiqueta_pred = id2label_base[int(np.argmax(salida))]
        y_true_3.append(MAPEO_3[item["etiqueta_final"]])
        y_pred_3.append(etiqueta_pred)

acc_zero_shot = accuracy_score(y_true_3, y_pred_3)
f1_zero_shot = f1_score(y_true_3, y_pred_3, labels=["contradiction"], average="macro", zero_division=0)
print("accuracy zero shot, tres clases:", acc_zero_shot)
print("f1 zero shot, clase contradiction:", f1_zero_shot)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

accuracy zero shot, tres clases: 0.4507042253521127
f1 zero shot, clase contradiction: 0.23529411764705882


## Guardar el modelo entrenado

Se guardan los adaptadores LoRA, no el modelo completo, ya que peft permite reconstruir el modelo cargando el modelo base y aplicando los adaptadores guardados.


In [19]:
RUTA_MODELO = "lora_5clases_multilingue"
trainer_final.model.save_pretrained(RUTA_MODELO)
tokenizer.save_pretrained(RUTA_MODELO)

import shutil
shutil.make_archive(RUTA_MODELO, "zip", RUTA_MODELO)

from google.colab import files as colab_files
colab_files.download(f"{RUTA_MODELO}.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from huggingface_hub import login
login()

trainer_final.model.push_to_hub("sinai1carlos/mdeberta-lora-nli-juridico-peru-5clases")
tokenizer.push_to_hub("sinai1carlos/mdeberta-lora-nli-juridico-peru-5clases")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  47%|####6     |  556kB / 1.19MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mptqk1kkk1/tokenizer.json:  10%|9         | 1.55MB / 16.0MB            

CommitInfo(commit_url='https://huggingface.co/sinai1carlos/mdeberta-lora-nli-juridico-peru-5clases/commit/756ef49807c3c97a2924a838ca31d0cce04bd0d3', commit_message='Upload tokenizer', commit_description='', oid='756ef49807c3c97a2924a838ca31d0cce04bd0d3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sinai1carlos/mdeberta-lora-nli-juridico-peru-5clases', endpoint='https://huggingface.co', repo_type='model', repo_id='sinai1carlos/mdeberta-lora-nli-juridico-peru-5clases'), pr_revision=None, pr_num=None)

## Funcion de inferencia

Funcion simple para clasificar un nuevo par de textos con el modelo ya entrenado, pensada como base para el modulo de despliegue del proyecto.


In [21]:
def clasificar_relacion(texto_a, texto_b, modelo=trainer_final.model, tok=tokenizer):
    modelo.eval()
    enc = tok(texto_a, texto_b, truncation=True, max_length=384, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = modelo(**enc).logits
        probs = logits.softmax(dim=-1)[0].cpu().numpy()
    etiqueta = ETIQUETAS[int(np.argmax(probs))]
    distribucion = {ETIQUETAS[i]: float(probs[i]) for i in range(5)}
    return etiqueta, distribucion

ejemplo_a = gold_data[0]["texto_a"]
ejemplo_b = gold_data[0]["texto_b"]
print(clasificar_relacion(ejemplo_a, ejemplo_b))
print("etiqueta real:", gold_data[0]["etiqueta_final"])


('EXCEPCION', {'CONTRADICCION': 0.15966796875, 'COMPATIBILIDAD': 0.22705078125, 'ESPECIFICACION': 0.1922607421875, 'EXCEPCION': 0.2919921875, 'NO_RELACION': 0.1290283203125})
etiqueta real: NO_RELACION
